In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right"],
                        description="Start corner",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length,tile,grout,reverse):
    """Physical positions of tile pieces, preserving tile orientation from chosen corner."""
    result=[]
    step=tile+grout
    n=math.ceil((length+grout)/step)
    if n>10000: raise ValueError("Too many tiles for this prototype.")
    for i in range(n):
        if reverse:
            full_end=length-i*step
            full_start=full_end-tile
        else:
            full_start=i*step
            full_end=full_start+tile
        start=max(0.0,full_start)
        end=min(length,full_end)
        if end>start+1e-8:
            result.append((start,end,start-full_start,full_end-end))
    return sorted(result)

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            rx=corner.value.endswith("right")
            ry=corner.value.startswith("Bottom")
            xs=segments(Wmm,Tw,G,rx)
            ys=segments(Hmm,Th,G,ry)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            edge_x=xs[0] if rx else xs[-1]
            edge_y=ys[0] if ry else ys[-1]
            cut_w=edge_x[1]-edge_x[0]
            cut_h=edge_y[1]-edge_y[0]
            edge_x_name="left" if rx else "right"
            edge_y_name="top" if ry else "bottom"
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"START: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            print(f"LAST TILE AT {edge_x_name.upper()} BOUNDARY: starts {edge_x[0]:g}, ends {edge_x[1]:g}, width {cut_w:g} mm")
            print(f"LAST TILE AT {edge_y_name.upper()} BOUNDARY: starts {edge_y[0]:g}, ends {edge_y[1]:g}, length {cut_h:g} mm")
            print(f"CORNER PIECE AT OPPOSITE EDGES: {cut_w:g} × {cut_h:g} mm")
            print("X tile ranges (mm):",", ".join(f"{a:g}–{b:g}" for a,b,*_ in xs))
            print("Y tile ranges (mm):",", ".join(f"{a:g}–{b:g}" for a,b,*_ in ys))
        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out
]))


In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right",
                                 "Centre tile (X and Y)","Centre grout (X and Y)",
                                 "Centre tile X / grout Y","Centre grout X / tile Y"],
                        description="Layout",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length, tile, grout, mode):
    """Physical tile ranges with texture offset, for corner or centered alignment."""
    pitch = tile + grout
    if mode == "start":
        origin = 0.0
    elif mode == "end":
        origin = length - tile
    elif mode == "centre_tile":
        origin = (length - tile) / 2
    elif mode == "centre_grout":
        origin = length / 2 + grout / 2
    else:
        raise ValueError("Unknown alignment")
    # Full tile start positions are origin + k*pitch.
    kmin = math.floor((-tile-origin)/pitch)+1
    kmax = math.ceil((length-origin)/pitch)-1
    if kmax-kmin > 10000:
        raise ValueError("Too many tiles.")
    result = []
    for k in range(kmin,kmax+1):
        a = origin+k*pitch
        b = a+tile
        left=max(0.,a);right=min(length,b)
        if right-left>1e-8:
            result.append((left,right,left-a,b-right))
    return sorted(result)

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            alignments = {
                "Top left": ("start", "start"),
                "Top right": ("end", "start"),
                "Bottom left": ("start", "end"),
                "Bottom right": ("end", "end"),
                "Centre tile (X and Y)": ("centre_tile", "centre_tile"),
                "Centre grout (X and Y)": ("centre_grout", "centre_grout"),
                "Centre tile X / grout Y": ("centre_tile", "centre_grout"),
                "Centre grout X / tile Y": ("centre_grout", "centre_tile")
            }
            ax_mode, ay_mode = alignments[corner.value]
            xs=segments(Wmm,Tw,G,ax_mode)
            ys=segments(Hmm,Th,G,ay_mode)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"LAYOUT: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            for axis_name, pieces, full in [("X", xs, Tw), ("Y", ys, Th)]:
                print(f"{axis_name} tile ranges (mm):", ", ".join(f"{a:g}–{b:g}" for a,b,*_ in pieces))
                for side, piece in [("first",pieces[0]),("last",pieces[-1])]:
                    width=piece[1]-piece[0]
                    if width<full-1e-7:
                        print(f"  {axis_name} {side} boundary cut: {width:g} mm (trim {full-width:g} mm)")
            print(f"Top-left piece: {xs[0][1]-xs[0][0]:g} × {ys[0][1]-ys[0][0]:g} mm")
            print(f"Bottom-right piece: {xs[-1][1]-xs[-1][0]:g} × {ys[-1][1]-ys[-1][0]:g} mm")
        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out
]))


In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right",
                                 "Top — centre tile","Top — centre grout",
                                 "Bottom — centre tile","Bottom — centre grout",
                                 "Left — centre tile","Left — centre grout",
                                 "Right — centre tile","Right — centre grout"],
                        description="Layout",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length, tile, grout, mode):
    """Physical tile ranges with texture offset, for corner or centered alignment."""
    pitch = tile + grout
    if mode == "start":
        origin = 0.0
    elif mode == "end":
        origin = length - tile
    elif mode == "centre_tile":
        origin = (length - tile) / 2
    elif mode == "centre_grout":
        origin = length / 2 + grout / 2
    else:
        raise ValueError("Unknown alignment")
    # Full tile start positions are origin + k*pitch.
    kmin = math.floor((-tile-origin)/pitch)+1
    kmax = math.ceil((length-origin)/pitch)-1
    if kmax-kmin > 10000:
        raise ValueError("Too many tiles.")
    result = []
    for k in range(kmin,kmax+1):
        a = origin+k*pitch
        b = a+tile
        left=max(0.,a);right=min(length,b)
        if right-left>1e-8:
            result.append((left,right,left-a,b-right))
    return sorted(result)

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            alignments = {
                "Top left": ("start", "start"),
                "Top right": ("end", "start"),
                "Bottom left": ("start", "end"),
                "Bottom right": ("end", "end"),
                "Top — centre tile": ("centre_tile", "start"),
                "Top — centre grout": ("centre_grout", "start"),
                "Bottom — centre tile": ("centre_tile", "end"),
                "Bottom — centre grout": ("centre_grout", "end"),
                "Left — centre tile": ("start", "centre_tile"),
                "Left — centre grout": ("start", "centre_grout"),
                "Right — centre tile": ("end", "centre_tile"),
                "Right — centre grout": ("end", "centre_grout")
            }
            ax_mode, ay_mode = alignments[corner.value]
            xs=segments(Wmm,Tw,G,ax_mode)
            ys=segments(Hmm,Th,G,ay_mode)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"LAYOUT: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            for axis_name, pieces, full in [("X", xs, Tw), ("Y", ys, Th)]:
                print(f"{axis_name} tile ranges (mm):", ", ".join(f"{a:g}–{b:g}" for a,b,*_ in pieces))
                for side, piece in [("first",pieces[0]),("last",pieces[-1])]:
                    width=piece[1]-piece[0]
                    if width<full-1e-7:
                        print(f"  {axis_name} {side} boundary cut: {width:g} mm (trim {full-width:g} mm)")
            print(f"Top-left piece: {xs[0][1]-xs[0][0]:g} × {ys[0][1]-ys[0][0]:g} mm")
            print(f"Bottom-right piece: {xs[-1][1]-xs[-1][0]:g} × {ys[-1][1]-ys[-1][0]:g} mm")
        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out
]))


In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right",
                                 "Top — centre tile","Top — centre grout",
                                 "Bottom — centre tile","Bottom — centre grout",
                                 "Left — centre tile","Left — centre grout",
                                 "Right — centre tile","Right — centre grout"],
                        description="Layout",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length, tile, grout, mode):
    """Physical tile ranges with texture offset, for corner or centered alignment."""
    pitch = tile + grout
    if mode == "start":
        origin = 0.0
    elif mode == "end":
        origin = length - tile
    elif mode == "centre_tile":
        origin = (length - tile) / 2
    elif mode == "centre_grout":
        origin = length / 2 + grout / 2
    else:
        raise ValueError("Unknown alignment")
    # Full tile start positions are origin + k*pitch.
    kmin = math.floor((-tile-origin)/pitch)+1
    kmax = math.ceil((length-origin)/pitch)-1
    if kmax-kmin > 10000:
        raise ValueError("Too many tiles.")
    result = []
    for k in range(kmin,kmax+1):
        a = origin+k*pitch
        b = a+tile
        left=max(0.,a);right=min(length,b)
        if right-left>1e-8:
            result.append((left,right,left-a,b-right))
    return sorted(result)


def tile_quantity(xs, ys, tile_w, tile_h):
    """Estimate whole tiles using reusable rectangular offcuts, without rotation.

    Each required piece is a rectangle. A piece can be cut from an existing
    rectangular offcut; leftover rectangles are returned to the stock.
    Assumes zero saw-blade kerf, no breakage, and matching tile orientation.
    """
    eps = 1e-6
    widths = [b-a for a,b,*_ in xs]
    heights = [b-a for a,b,*_ in ys]
    pieces = [(w,h) for h in heights for w in widths]
    full = sum(w >= tile_w-eps and h >= tile_h-eps for w,h in pieces)
    cuts = [(w,h) for w,h in pieces
            if not (w >= tile_w-eps and h >= tile_h-eps)]
    cuts.sort(key=lambda wh: (wh[0]*wh[1],max(wh)),reverse=True)
    stock = []  # reusable rectangular offcuts: (width, height)
    new_tiles = 0
    for w,h in cuts:
        fits = [(i, (sw*sh-w*h))
                for i,(sw,sh) in enumerate(stock)
                if sw+eps >= w and sh+eps >= h]
        if fits:
            idx = min(fits,key=lambda t:t[1])[0]
            sw,sh=stock.pop(idx)
        else:
            sw,sh=tile_w,tile_h
            new_tiles+=1
        # Choose the guillotine cut order leaving the larger useful rectangles.
        if (sw-w)*sh >= sw*(sh-h):
            leftovers=[(sw-w,sh),(w,sh-h)]
        else:
            leftovers=[(sw,sh-h),(sw-w,h)]
        stock.extend((a,b) for a,b in leftovers if a>eps and b>eps)
    total=full+new_tiles
    area=sum(w*h for w,h in pieces)
    waste=max(0.,total*tile_w*tile_h-area)
    return total,full,new_tiles,len(cuts),waste,area

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            alignments = {
                "Top left": ("start", "start"),
                "Top right": ("end", "start"),
                "Bottom left": ("start", "end"),
                "Bottom right": ("end", "end"),
                "Top — centre tile": ("centre_tile", "start"),
                "Top — centre grout": ("centre_grout", "start"),
                "Bottom — centre tile": ("centre_tile", "end"),
                "Bottom — centre grout": ("centre_grout", "end"),
                "Left — centre tile": ("start", "centre_tile"),
                "Left — centre grout": ("start", "centre_grout"),
                "Right — centre tile": ("end", "centre_tile"),
                "Right — centre grout": ("end", "centre_grout")
            }
            ax_mode, ay_mode = alignments[corner.value]
            xs=segments(Wmm,Tw,G,ax_mode)
            ys=segments(Hmm,Th,G,ay_mode)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"LAYOUT: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            for axis_name, pieces, full in [("X", xs, Tw), ("Y", ys, Th)]:
                print(f"{axis_name} tile ranges (mm):", ", ".join(f"{a:g}–{b:g}" for a,b,*_ in pieces))
                for side, piece in [("first",pieces[0]),("last",pieces[-1])]:
                    width=piece[1]-piece[0]
                    if width<full-1e-7:
                        print(f"  {axis_name} {side} boundary cut: {width:g} mm (trim {full-width:g} mm)")
            print(f"Top-left piece: {xs[0][1]-xs[0][0]:g} × {ys[0][1]-ys[0][0]:g} mm")
            print(f"Bottom-right piece: {xs[-1][1]-xs[-1][0]:g} × {ys[-1][1]-ys[-1][0]:g} mm")
            total, full, cut_tiles, cut_pieces, waste, covered = tile_quantity(xs,ys,Tw,Th)
            print("\\nTILE QUANTITY (reusing suitable rectangular offcuts)")
            print(f"Full uncut tiles: {full}")
            print(f"Cut pieces required: {cut_pieces}")
            print(f"Additional whole tiles for cuts: {cut_tiles}")
            print(f"TOTAL WHOLE TILES NEEDED: {total}")
            print(f"Estimated leftover tile material: {waste/1_000_000:.3f} m²")
            print("Estimate assumes no cutting-blade loss, no breakage, and no tile rotation.")
            print("Offcuts are reused only when their actual width AND length fit;")
            print("two pieces under half-area do not necessarily fit on one tile.")

        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out

]))


In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right",
                                 "Top — centre tile","Top — centre grout",
                                 "Bottom — centre tile","Bottom — centre grout",
                                 "Left — centre tile","Left — centre grout",
                                 "Right — centre tile","Right — centre grout"],
                        description="Layout",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length, tile, grout, mode):
    """Physical tile ranges with texture offset, for corner or centered alignment."""
    pitch = tile + grout
    if mode == "start":
        origin = 0.0
    elif mode == "end":
        origin = length - tile
    elif mode == "centre_tile":
        origin = (length - tile) / 2
    elif mode == "centre_grout":
        origin = length / 2 + grout / 2
    else:
        raise ValueError("Unknown alignment")
    # Full tile start positions are origin + k*pitch.
    kmin = math.floor((-tile-origin)/pitch)+1
    kmax = math.ceil((length-origin)/pitch)-1
    if kmax-kmin > 10000:
        raise ValueError("Too many tiles.")
    result = []
    for k in range(kmin,kmax+1):
        a = origin+k*pitch
        b = a+tile
        left=max(0.,a);right=min(length,b)
        if right-left>1e-8:
            result.append((left,right,left-a,b-right))
    return sorted(result)


def tile_quantity(xs, ys, tile_w, tile_h, image, tolerance=24):
    """Greedy physical offcut allocation, with pixel matching at 0° and 180°.

    Candidate offcuts retain original image coordinates. Only the *visible*
    cropped region is compared; unused/missing pixels are never compared.
    No mirroring, no resizing a partial tile to a full tile.
    """
    import numpy as np
    eps=1e-6
    # Work in original tile coordinates. One pixel grid for each tile dimension.
    # Resampling the FULL tile once keeps cut comparisons consistent.
    resolution=320
    rw=resolution
    rh=max(1,round(resolution*tile_h/tile_w))
    reference=np.asarray(image.convert("RGB").resize((rw,rh),Image.Resampling.BILINEAR),
                         dtype=np.int16)

    def pixel_crop(x,y,w,h):
        x0=max(0,min(rw-1,round(x/tile_w*rw)))
        y0=max(0,min(rh-1,round(y/tile_h*rh)))
        x1=max(x0+1,min(rw,round((x+w)/tile_w*rw)))
        y1=max(y0+1,min(rh,round((y+h)/tile_h*rh)))
        return reference[y0:y1,x0:x1]

    def compare(px,py,w,h,target,angle):
        candidate=pixel_crop(px,py,w,h)
        if angle==180:
            candidate=candidate[::-1,::-1]
        # Small coordinate rounding differences: align only the visible crop.
        if candidate.shape != target.shape:
            from PIL import Image as PILImage
            candidate=np.asarray(PILImage.fromarray(candidate.astype("uint8")).resize(
                (target.shape[1],target.shape[0]),PILImage.Resampling.BILINEAR),
                dtype=np.int16)
        return float(np.mean(np.abs(candidate-target)))

    pieces=[(x1-x0,y1-y0,xc,yc)
            for y0,y1,yc,_ in ys for x0,x1,xc,_ in xs]
    full=sum(w>=tile_w-eps and h>=tile_h-eps for w,h,_,_ in pieces)
    cuts=sorted((p for p in pieces if p[0]<tile_w-eps or p[1]<tile_h-eps),
                key=lambda p:p[0]*p[1],reverse=True)

    # Free rectangles: (tile number, original x, original y, width, height)
    free=[]
    new_tiles=reused=rotated=0

    def candidate_positions(fx,fy,fw,fh,w,h,tx,ty):
        """Check matching source locations and edge/corner positions."""
        # Exact source location and its 180° counterpart.
        positions=[(tx,ty),(tile_w-tx-w,tile_h-ty-h)]
        # Also try the corners of the available material, where real cuts
        # are most practical and preserve useful rectangular offcuts.
        for x in (fx,fx+fw-w):
            for y in (fy,fy+fh-h):
                positions.append((x,y))
        # Sample additional source locations for repeating motifs.
        # 24 samples per axis maximum; always include exact boundaries.
        nx=min(24,max(1,math.ceil((fw-w)/max(tile_w/40,1))))
        ny=min(24,max(1,math.ceil((fh-h)/max(tile_h/40,1))))
        for ix in range(nx+1):
            x=fx+(fw-w)*ix/nx
            for iy in range(ny+1):
                y=fy+(fh-h)*iy/ny
                positions.append((x,y))
        seen=set()
        for x,y in positions:
            if x<fx-eps or y<fy-eps or x+w>fx+fw+eps or y+h>fy+fh+eps:
                continue
            key=(round(x,5),round(y,5))
            if key not in seen:
                seen.add(key)
                yield x,y

    for w,h,tx,ty in cuts:
        target=pixel_crop(tx,ty,w,h)
        best=None
        for idx,(_,fx,fy,fw,fh) in enumerate(free):
            if fw+eps<w or fh+eps<h:
                continue
            for px,py in candidate_positions(fx,fy,fw,fh,w,h,tx,ty):
                for angle in (0,180):
                    err=compare(px,py,w,h,target,angle)
                    if err<=tolerance:
                        # Prefer using existing stock; preserve large remnants.
                        key=(err,fw*fh-w*h)
                        if best is None or key<best[0]:
                            best=(key,idx,px,py,angle)
                        if err<0.05:
                            break
        if best is None:
            new_tiles+=1
            fx=fy=0.0;fw=tile_w;fh=tile_h
            # New tile always has the exact source pattern required.
            px,py,angle=tx,ty,0
            tile_id=new_tiles
        else:
            _,idx,px,py,angle=best
            tile_id,fx,fy,fw,fh=free.pop(idx)
            reused+=1
        rotated+=int(angle==180)
        left=px-fx; right=fx+fw-px-w
        top=py-fy; bottom=fy+fh-py-h
        vertical=[(fx,fy,left,fh),(px+w,fy,right,fh),
                  (px,fy,w,top),(px,py+h,w,bottom)]
        horizontal=[(fx,fy,fw,top),(fx,py+h,fw,bottom),
                    (fx,py,left,h),(px+w,py,right,h)]
        # Prefer a split that preserves the largest remaining rectangle.
        chosen=max((vertical,horizontal),
                   key=lambda r:max((c*d for _,_,c,d in r),default=0))
        free.extend((tile_id,x,y,c,d) for x,y,c,d in chosen if c>eps and d>eps)

    total=full+new_tiles
    covered=sum(w*h for w,h,_,_ in pieces)
    waste=max(0,total*tile_w*tile_h-covered)
    return total,full,new_tiles,len(cuts),waste,covered,rotated,reused

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            alignments = {
                "Top left": ("start", "start"),
                "Top right": ("end", "start"),
                "Bottom left": ("start", "end"),
                "Bottom right": ("end", "end"),
                "Top — centre tile": ("centre_tile", "start"),
                "Top — centre grout": ("centre_grout", "start"),
                "Bottom — centre tile": ("centre_tile", "end"),
                "Bottom — centre grout": ("centre_grout", "end"),
                "Left — centre tile": ("start", "centre_tile"),
                "Left — centre grout": ("start", "centre_grout"),
                "Right — centre tile": ("end", "centre_tile"),
                "Right — centre grout": ("end", "centre_grout")
            }
            ax_mode, ay_mode = alignments[corner.value]
            xs=segments(Wmm,Tw,G,ax_mode)
            ys=segments(Hmm,Th,G,ay_mode)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"LAYOUT: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            for axis_name, pieces, full in [("X", xs, Tw), ("Y", ys, Th)]:
                print(f"{axis_name} tile ranges (mm):", ", ".join(f"{a:g}–{b:g}" for a,b,*_ in pieces))
                for side, piece in [("first",pieces[0]),("last",pieces[-1])]:
                    width=piece[1]-piece[0]
                    if width<full-1e-7:
                        print(f"  {axis_name} {side} boundary cut: {width:g} mm (trim {full-width:g} mm)")
            print(f"Top-left piece: {xs[0][1]-xs[0][0]:g} × {ys[0][1]-ys[0][0]:g} mm")
            print(f"Bottom-right piece: {xs[-1][1]-xs[-1][0]:g} × {ys[-1][1]-ys[-1][0]:g} mm")
            total, full, cut_tiles, cut_pieces, waste, covered, rotated, reused = tile_quantity(xs,ys,Tw,Th,tile)
            print("\nPATTERN-AWARE TILE QUANTITY (including 180° rotations)")
            print(f"Full uncut tiles: {full}")
            print(f"Cut pieces required: {cut_pieces}")
            print(f"Additional whole tiles for cuts: {cut_tiles}")
            print(f"Cut pieces taken from existing offcuts: {reused}")
            print(f"Pieces installed after 180° rotation: {rotated}")
            print(f"TOTAL WHOLE TILES NEEDED: {total}")
            print(f"Estimated leftover tile material: {waste/1_000_000:.3f} m²")
            print("Estimate assumes no blade loss or breakage; 0° and 180° rotations allowed.")
            print("Offcuts must fit physically AND match the visible pattern.")
            print("Pattern tolerance: 24/255 mean RGB difference (JPEG imperfections allowed).")
            print("This is a cutting estimate, not an optimized or guaranteed minimum.")

        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out
]))


In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right",
                                 "Top — centre tile","Top — centre grout",
                                 "Bottom — centre tile","Bottom — centre grout",
                                 "Left — centre tile","Left — centre grout",
                                 "Right — centre tile","Right — centre grout"],
                        description="Layout",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length, tile, grout, mode):
    """Physical tile ranges with texture offset, for corner or centered alignment."""
    pitch = tile + grout
    if mode == "start":
        origin = 0.0
    elif mode == "end":
        origin = length - tile
    elif mode == "centre_tile":
        origin = (length - tile) / 2
    elif mode == "centre_grout":
        origin = length / 2 + grout / 2
    else:
        raise ValueError("Unknown alignment")
    # Full tile start positions are origin + k*pitch.
    kmin = math.floor((-tile-origin)/pitch)+1
    kmax = math.ceil((length-origin)/pitch)-1
    if kmax-kmin > 10000:
        raise ValueError("Too many tiles.")
    result = []
    for k in range(kmin,kmax+1):
        a = origin+k*pitch
        b = a+tile
        left=max(0.,a);right=min(length,b)
        if right-left>1e-8:
            result.append((left,right,left-a,b-right))
    return sorted(result)


def tile_quantity(xs, ys, tile_w, tile_h, image, tolerance=24):
    """Beam-search cutting planner: jointly considers multiple offcuts.

    Pixel checks use only the installed region. Each stock rectangle retains
    coordinates on its original tile. A piece can be turned 180°, not mirrored.
    The search tries several cutting plans rather than committing greedily.
    This is still an estimate, not a proof of global optimality.
    """
    import numpy as np
    from PIL import Image as PILImage

    eps = 1e-5
    # Fixed full-tile image grid; physical positions map to these coordinates.
    rw = 250
    rh = max(1, round(rw * tile_h / tile_w))
    ref = np.asarray(image.convert("RGB").resize(
        (rw, rh), PILImage.Resampling.BILINEAR), dtype=np.int16)

    def crop(x, y, w, h):
        x0 = max(0, min(rw - 1, round(x / tile_w * rw)))
        y0 = max(0, min(rh - 1, round(y / tile_h * rh)))
        x1 = max(x0 + 1, min(rw, round((x + w) / tile_w * rw)))
        y1 = max(y0 + 1, min(rh, round((y + h) / tile_h * rh)))
        return ref[y0:y1, x0:x1]

    def error(x, y, w, h, target, angle):
        part = crop(x, y, w, h)
        if angle == 180:
            part = part[::-1, ::-1]
        if part.shape != target.shape:
            part = np.asarray(PILImage.fromarray(part.astype("uint8")).resize(
                (target.shape[1], target.shape[0]),
                PILImage.Resampling.BILINEAR), dtype=np.int16)
        return float(np.mean(np.abs(part - target)))

    pieces = [(x1-x0, y1-y0, xc, yc, x0, y0)
              for y0,y1,yc,_ in ys for x0,x1,xc,_ in xs]
    full = sum(w >= tile_w-eps and h >= tile_h-eps for w,h,*_ in pieces)
    cuts = [p for p in pieces if p[0]<tile_w-eps or p[1]<tile_h-eps]
    # Largest pieces first is useful for packing, but several packing states
    # are retained at every step (beam search).
    cuts.sort(key=lambda p: (p[0]*p[1], max(p[:2])), reverse=True)

    def split(rect, px, py, w, h):
        tid, fx, fy, fw, fh = rect
        left, right = px-fx, fx+fw-px-w
        top, bottom = py-fy, fy+fh-py-h
        v = [(tid,fx,fy,left,fh),(tid,px+w,fy,right,fh),
             (tid,px,fy,w,top),(tid,px,py+h,w,bottom)]
        hsplit = [(tid,fx,fy,fw,top),(tid,fx,py+h,fw,bottom),
                  (tid,fx,py,left,h),(tid,px+w,py,right,h)]
        return [tuple(r for r in option if r[3]>eps and r[4]>eps)
                for option in (v,hsplit)]

    def placements(rect, w, h, tx, ty, target):
        _,fx,fy,fw,fh = rect
        if fw+eps<w or fh+eps<h:
            return []
        # Try exact reference position, its rotated counterpart, stock corners,
        # and a small grid for repeating patterns.
        coords = [(tx,ty), (tile_w-tx-w,tile_h-ty-h)]
        coords += [(x,y) for x in (fx,fx+fw-w)
                   for y in (fy,fy+fh-h)]
        # Pixel-accurate grid across each offcut, bounded for notebook speed.
        nx = min(8,max(1,round((fw-w)/max(tile_w/12,1))))
        ny = min(8,max(1,round((fh-h)/max(tile_h/12,1))))
        coords += [(fx+(fw-w)*i/nx,fy+(fh-h)*j/ny)
                   for i in range(nx+1) for j in range(ny+1)]
        matches = []
        seen = set()
        for px,py in coords:
            if px<fx-eps or py<fy-eps or px+w>fx+fw+eps or py+h>fy+fh+eps:
                continue
            key=(round(px,4),round(py,4))
            if key in seen:
                continue
            seen.add(key)
            for angle in (0,180):
                score=error(px,py,w,h,target,angle)
                if score <= tolerance:
                    matches.append((score,px,py,angle))
        matches.sort()
        # Keep alternative cuts, not just the first valid match.
        return matches[:8]

    # State: (new tile count, reuse count, rotations, free rectangles,
    #         cutting instructions, accumulated pattern error)
    states = [(0,0,0,tuple(),tuple(),0.)]
    beam_width = 70
    for w,h,tx,ty,dx,dy in cuts:
        target=crop(tx,ty,w,h)
        next_states=[]
        for new,reuse,rot,free,plan,score in states:
            # Existing stock first, but retain alternatives.
            for idx,rect in enumerate(free):
                for e,px,py,angle in placements(rect,w,h,tx,ty,target):
                    for remnants in split(rect,px,py,w,h):
                        updated=free[:idx]+free[idx+1:]+remnants
                        step=(rect[0],round(px,2),round(py,2),
                              round(w,2),round(h,2),angle)
                        next_states.append((new,reuse+1,rot+int(angle==180),
                                            updated,plan+(step,),score+e))
            # A fresh tile is always a valid fallback, and must be allowed
            # even if an offcut also fits (to improve later cuts).
            rect=(new+1,0.,0.,tile_w,tile_h)
            for remnants in split(rect,tx,ty,w,h):
                step=(new+1,round(tx,2),round(ty,2),
                      round(w,2),round(h,2),0)
                next_states.append((new+1,reuse,rot,free+remnants,
                                    plan+(step,),score))
        # Prefer fewer whole tiles, then more reusable stock area and accuracy.
        next_states.sort(key=lambda st:(
            st[0], -max((r[3]*r[4] for r in st[3]),default=0),
            st[5], -st[1]))
        # Deduplicate equivalent material states.
        unique={}
        for st in next_states:
            signature=(st[0],tuple(sorted(
                (r[0],round(r[1],1),round(r[2],1),
                 round(r[3],1),round(r[4],1)) for r in st[3])))
            if signature not in unique:
                unique[signature]=st
            if len(unique)>=beam_width:
                break
        states=list(unique.values())
    new,reused,rotated,free,plan,score=min(states,key=lambda st:(st[0],st[5]))
    total=full+new
    covered=sum(w*h for w,h,*_ in pieces)
    waste=max(0.,total*tile_w*tile_h-covered)
    # Same order as the cutting plan, with physical destination coordinates.
    destinations=tuple((dx,dy,w,h) for w,h,tx,ty,dx,dy in cuts)
    return total,full,new,len(cuts),waste,covered,rotated,reused,plan,destinations

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            alignments = {
                "Top left": ("start", "start"),
                "Top right": ("end", "start"),
                "Bottom left": ("start", "end"),
                "Bottom right": ("end", "end"),
                "Top — centre tile": ("centre_tile", "start"),
                "Top — centre grout": ("centre_grout", "start"),
                "Bottom — centre tile": ("centre_tile", "end"),
                "Bottom — centre grout": ("centre_grout", "end"),
                "Left — centre tile": ("start", "centre_tile"),
                "Left — centre grout": ("start", "centre_grout"),
                "Right — centre tile": ("end", "centre_tile"),
                "Right — centre grout": ("end", "centre_grout")
            }
            ax_mode, ay_mode = alignments[corner.value]
            xs=segments(Wmm,Tw,G,ax_mode)
            ys=segments(Hmm,Th,G,ay_mode)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            # Compute the plan BEFORE drawing labels so piece numbers correspond
            # exactly to the numbered cutting instructions.
            total, full, cut_tiles, cut_pieces, waste, covered, rotated, reused, plan, destinations = tile_quantity(xs,ys,Tw,Th,tile)
            for number,(dx,dy,pw,ph) in enumerate(destinations,1):
                ax.text(dx+pw/2,dy+ph/2,f"C{number}",
                        ha="center",va="center",fontsize=8,fontweight="bold",
                        color="black",clip_on=True,
                        bbox=dict(facecolor="white",edgecolor="black",
                                  alpha=0.88,boxstyle="round,pad=0.18"))
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"LAYOUT: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            for axis_name, pieces, full in [("X", xs, Tw), ("Y", ys, Th)]:
                print(f"{axis_name} tile ranges (mm):", ", ".join(f"{a:g}–{b:g}" for a,b,*_ in pieces))
                for side, piece in [("first",pieces[0]),("last",pieces[-1])]:
                    width=piece[1]-piece[0]
                    if width<full-1e-7:
                        print(f"  {axis_name} {side} boundary cut: {width:g} mm (trim {full-width:g} mm)")
            print(f"Top-left piece: {xs[0][1]-xs[0][0]:g} × {ys[0][1]-ys[0][0]:g} mm")
            print(f"Bottom-right piece: {xs[-1][1]-xs[-1][0]:g} × {ys[-1][1]-ys[-1][0]:g} mm")
            print("\nPATTERN-AWARE TILE QUANTITY (including 180° rotations)")
            print(f"Full uncut tiles: {full}")
            print(f"Cut pieces required: {cut_pieces}")
            print(f"Additional whole tiles for cuts: {cut_tiles}")
            print(f"Cut pieces taken from existing offcuts: {reused}")
            print(f"Pieces installed after 180° rotation: {rotated}")
            print(f"TOTAL WHOLE TILES NEEDED: {total}")
            print(f"Estimated leftover tile material: {waste/1_000_000:.3f} m²")
            print("Estimate assumes no blade loss or breakage; 0° and 180° rotations allowed.")
            print("Offcuts must fit physically AND match the visible pattern.")
            print("Pattern tolerance: 24/255 mean RGB difference (JPEG imperfections allowed).")
            print("CUTTING PLAN (tile number among additional cut tiles; source x/y in mm):")
            for i,((tid,px,py,pw,ph,angle),(dx,dy,dw,dh)) in enumerate(zip(plan,destinations),1):
                print(f"  C{i}: source tile {tid} at ({px:g},{py:g}) mm, "
                      f"cut {pw:g} × {ph:g} mm, rotate {angle}°, "
                      f"INSTALL at floor ({dx:g},{dy:g}) mm")
            print("This is a cutting estimate, not an optimized or guaranteed minimum.")

        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out
]))


In [ ]:
# Tile Visualizer: Jupyter GUI, grout, readable measurements, four starting corners.
# If needed, run separately: %pip install pillow matplotlib ipywidgets
import io, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output

def measure(label, default, unit):
    n = widgets.FloatText(value=default, description=label, style={"description_width":"110px"})
    u = widgets.Dropdown(options=["mm","cm","m","in","ft"], value=unit,
                         layout=widgets.Layout(width="85px"))
    return n,u,widgets.HBox([n,u])

fw,fwu,fwui=measure("Floor width",2,"m")
fh,fhu,fhui=measure("Floor length",2,"m")
tw,twu,twui=measure("Tile width",600,"mm")
th,thu,thui=measure("Tile length",600,"mm")
gw,gwu,gwui=measure("Grout width",3,"mm")
color=widgets.ColorPicker(value="#b0b0b0",description="Grout colour",
                          style={"description_width":"110px"})
corner=widgets.Dropdown(options=["Top left","Top right","Bottom left","Bottom right",
                                 "Top — centre tile","Top — centre grout",
                                 "Bottom — centre tile","Bottom — centre grout",
                                 "Left — centre tile","Left — centre grout",
                                 "Right — centre tile","Right — centre grout"],
                        description="Layout",style={"description_width":"110px"})
upload=widgets.FileUpload(accept="image/*",multiple=False,description="Select Tile Image")
button=widgets.Button(description="Generate",button_style="success")
out=widgets.Output()

def mm(n,u):
    return float(n.value)*{"mm":1,"cm":10,"m":1000,"in":25.4,"ft":304.8}[u.value]

def get_image():
    val=upload.value
    if not val: raise ValueError("Select a tile image.")
    entry=next(iter(val.values())) if isinstance(val,dict) else val[0]
    return Image.open(io.BytesIO(bytes(entry["content"]))).convert("RGB")

def segments(length, tile, grout, mode):
    """Physical tile ranges with texture offset, for corner or centered alignment."""
    pitch = tile + grout
    if mode == "start":
        origin = 0.0
    elif mode == "end":
        origin = length - tile
    elif mode == "centre_tile":
        origin = (length - tile) / 2
    elif mode == "centre_grout":
        origin = length / 2 + grout / 2
    else:
        raise ValueError("Unknown alignment")
    # Full tile start positions are origin + k*pitch.
    kmin = math.floor((-tile-origin)/pitch)+1
    kmax = math.ceil((length-origin)/pitch)-1
    if kmax-kmin > 10000:
        raise ValueError("Too many tiles.")
    result = []
    for k in range(kmin,kmax+1):
        a = origin+k*pitch
        b = a+tile
        left=max(0.,a);right=min(length,b)
        if right-left>1e-8:
            result.append((left,right,left-a,b-right))
    return sorted(result)


def tile_quantity(xs, ys, tile_w, tile_h, image, tolerance=24):
    """Beam-search cutting planner: jointly considers multiple offcuts.

    Pixel checks use only the installed region. Each stock rectangle retains
    coordinates on its original tile. A piece can be turned 180°, not mirrored.
    The search tries several cutting plans rather than committing greedily.
    This is still an estimate, not a proof of global optimality.
    """
    import numpy as np
    from PIL import Image as PILImage

    eps = 1e-5
    # Fixed full-tile image grid; physical positions map to these coordinates.
    rw = 250
    rh = max(1, round(rw * tile_h / tile_w))
    ref = np.asarray(image.convert("RGB").resize(
        (rw, rh), PILImage.Resampling.BILINEAR), dtype=np.int16)

    def crop(x, y, w, h):
        x0 = max(0, min(rw - 1, round(x / tile_w * rw)))
        y0 = max(0, min(rh - 1, round(y / tile_h * rh)))
        x1 = max(x0 + 1, min(rw, round((x + w) / tile_w * rw)))
        y1 = max(y0 + 1, min(rh, round((y + h) / tile_h * rh)))
        return ref[y0:y1, x0:x1]

    def error(x, y, source_w, source_h, target, angle):
        part = crop(x, y, source_w, source_h)
        # np.rot90 uses counterclockwise rotations; 90/270 are now supported.
        part = np.rot90(part, k=angle // 90)
        if part.shape != target.shape:
            part = np.asarray(PILImage.fromarray(part.astype("uint8")).resize(
                (target.shape[1], target.shape[0]),
                PILImage.Resampling.BILINEAR), dtype=np.int16)
        return float(np.mean(np.abs(part - target)))

    pieces = [(x1-x0, y1-y0, xc, yc, x0, y0)
              for y0,y1,yc,_ in ys for x0,x1,xc,_ in xs]
    full = sum(w >= tile_w-eps and h >= tile_h-eps for w,h,*_ in pieces)
    cuts = [p for p in pieces if p[0]<tile_w-eps or p[1]<tile_h-eps]
    # Largest pieces first is useful for packing, but several packing states
    # are retained at every step (beam search).
    cuts.sort(key=lambda p: (p[0]*p[1], max(p[:2])), reverse=True)

    def split(rect, px, py, w, h):
        tid, fx, fy, fw, fh = rect
        left, right = px-fx, fx+fw-px-w
        top, bottom = py-fy, fy+fh-py-h
        v = [(tid,fx,fy,left,fh),(tid,px+w,fy,right,fh),
             (tid,px,fy,w,top),(tid,px,py+h,w,bottom)]
        hsplit = [(tid,fx,fy,fw,top),(tid,fx,py+h,fw,bottom),
                  (tid,fx,py,left,h),(tid,px+w,py,right,h)]
        return [tuple(r for r in option if r[3]>eps and r[4]>eps)
                for option in (v,hsplit)]

    def placements(rect, w, h, tx, ty, target):
        _,fx,fy,fw,fh = rect
        matches = []
        for angle in (0, 90, 180, 270):
            # Dimensions BEFORE rotation: 90/270 swap width and height.
            sw, sh = (h, w) if angle in (90,270) else (w,h)
            if fw+eps < sw or fh+eps < sh:
                continue
            # Exact inverse-rotation reference coordinates are always tried.
            if angle == 0:
                reference = (tx,ty)
            elif angle == 90:
                reference = (tile_w-ty-sh,tx)
            elif angle == 180:
                reference = (tile_w-tx-sw,tile_h-ty-sh)
            else:
                reference = (ty,tile_h-tx-sw)
            coords = [reference,(tx,ty),
                      (tile_w-tx-sw,tile_h-ty-sh)]
            coords += [(x,y) for x in (fx,fx+fw-sw)
                       for y in (fy,fy+fh-sh)]
            nx = min(8,max(1,round((fw-sw)/max(tile_w/12,1))))
            ny = min(8,max(1,round((fh-sh)/max(tile_h/12,1))))
            coords += [(fx+(fw-sw)*i/nx,fy+(fh-sh)*j/ny)
                       for i in range(nx+1) for j in range(ny+1)]
            seen = set()
            for px,py in coords:
                if (px < fx-eps or py < fy-eps or
                    px+sw > fx+fw+eps or py+sh > fy+fh+eps):
                    continue
                key = (round(px,4),round(py,4))
                if key in seen:
                    continue
                seen.add(key)
                score = error(px,py,sw,sh,target,angle)
                if score <= tolerance:
                    matches.append((score,px,py,angle,sw,sh))
        matches.sort()
        return matches[:12]

    # State: (new tile count, reuse count, rotations, free rectangles,
    #         cutting instructions, accumulated pattern error)
    states = [(0,0,0,tuple(),tuple(),0.)]
    beam_width = 70
    for w,h,tx,ty,dx,dy in cuts:
        target=crop(tx,ty,w,h)
        next_states=[]
        for new,reuse,rot,free,plan,score in states:
            # Existing stock first, but retain alternatives.
            for idx,rect in enumerate(free):
                for e,px,py,angle,sw,sh in placements(rect,w,h,tx,ty,target):
                    for remnants in split(rect,px,py,sw,sh):
                        updated=free[:idx]+free[idx+1:]+remnants
                        step=(rect[0],round(px,2),round(py,2),
                              round(sw,2),round(sh,2),angle)
                        next_states.append((new,reuse+1,rot+int(angle!=0),
                                            updated,plan+(step,),score+e))
            # A fresh tile is always a valid fallback, and must be allowed
            # even if an offcut also fits (to improve later cuts).
            rect=(new+1,0.,0.,tile_w,tile_h)
            for remnants in split(rect,tx,ty,w,h):
                step=(new+1,round(tx,2),round(ty,2),
                      round(w,2),round(h,2),0)
                next_states.append((new+1,reuse,rot,free+remnants,
                                    plan+(step,),score))
        # Prefer fewer whole tiles, then more reusable stock area and accuracy.
        next_states.sort(key=lambda st:(
            st[0], -max((r[3]*r[4] for r in st[3]),default=0),
            st[5], -st[1]))
        # Deduplicate equivalent material states.
        unique={}
        for st in next_states:
            signature=(st[0],tuple(sorted(
                (r[0],round(r[1],1),round(r[2],1),
                 round(r[3],1),round(r[4],1)) for r in st[3])))
            if signature not in unique:
                unique[signature]=st
            if len(unique)>=beam_width:
                break
        states=list(unique.values())
    new,reused,rotated,free,plan,score=min(states,key=lambda st:(st[0],st[5]))
    total=full+new
    covered=sum(w*h for w,h,*_ in pieces)
    waste=max(0.,total*tile_w*tile_h-covered)
    # Same order as the cutting plan, with physical destination coordinates.
    destinations=tuple((dx,dy,w,h) for w,h,tx,ty,dx,dy in cuts)
    return total,full,new,len(cuts),waste,covered,rotated,reused,plan,destinations

def render(_):
    with out:
        clear_output(wait=True)
        try:
            tile=get_image()
            Wmm,Hmm=mm(fw,fwu),mm(fh,fhu)
            Tw,Th=mm(tw,twu),mm(th,thu)
            G=mm(gw,gwu)
            if min(Wmm,Hmm,Tw,Th)<=0 or G<0: raise ValueError("Invalid measurements.")
            alignments = {
                "Top left": ("start", "start"),
                "Top right": ("end", "start"),
                "Bottom left": ("start", "end"),
                "Bottom right": ("end", "end"),
                "Top — centre tile": ("centre_tile", "start"),
                "Top — centre grout": ("centre_grout", "start"),
                "Bottom — centre tile": ("centre_tile", "end"),
                "Bottom — centre grout": ("centre_grout", "end"),
                "Left — centre tile": ("start", "centre_tile"),
                "Left — centre grout": ("start", "centre_grout"),
                "Right — centre tile": ("end", "centre_tile"),
                "Right — centre grout": ("end", "centre_grout")
            }
            ax_mode, ay_mode = alignments[corner.value]
            xs=segments(Wmm,Tw,G,ax_mode)
            ys=segments(Hmm,Th,G,ay_mode)
            scale=min(1.5,2400/max(Wmm,Hmm))
            W=max(1,round(Wmm*scale)); H=max(1,round(Hmm*scale))
            canvas=Image.new("RGB",(W,H),color.value)
            tW=max(1,round(Tw*scale));tH=max(1,round(Th*scale))
            texture=tile.resize((tW,tH),Image.Resampling.LANCZOS)
            for y0,y1,ycut0,ycut1 in ys:
                py0=round(y0*scale);py1=round(y1*scale)
                for x0,x1,xcut0,xcut1 in xs:
                    px0=round(x0*scale);px1=round(x1*scale)
                    if px1<=px0 or py1<=py0: continue
                    # Crop from the corresponding edge of the original tile, rather
                    # than restarting its pattern when anchoring to right/bottom.
                    sx=round(xcut0/Tw*tW);sy=round(ycut0/Th*tH)
                    crop=texture.crop((sx,sy,min(tW,sx+px1-px0),min(tH,sy+py1-py0)))
                    canvas.paste(crop,(px0,py0))
            # Avoid overlapping tick labels: use tile *starts* only, plus floor endpoint.
            # Display grout measurements in a separate summary.
            xticks=sorted(set([0.0,Wmm]+[s[0] for s in xs]))
            yticks=sorted(set([0.0,Hmm]+[s[0] for s in ys]))
            fig,ax=plt.subplots(figsize=(11,8))
            ax.imshow(canvas,extent=(0,Wmm,Hmm,0))
            ax.set_xlim(0,Wmm);ax.set_ylim(Hmm,0)
            ax.set_xticks(xticks);ax.set_yticks(yticks)
            ax.set_xticklabels([f"{x:g}" for x in xticks],rotation=45,ha="right")
            ax.set_yticklabels([f"{y:g}" for y in yticks])
            ax.tick_params(axis="both",labelsize=9)
            ax.set_xlabel("Floor width (mm)")
            ax.set_ylabel("Floor length (mm)")
            ax.set_title(f"Tile layout — {corner.value} start | grout {G:g} mm")
            # Compute the plan BEFORE drawing labels so piece numbers correspond
            # exactly to the numbered cutting instructions.
            total, full, cut_tiles, cut_pieces, waste, covered, rotated, reused, plan, destinations = tile_quantity(xs,ys,Tw,Th,tile)
            for number,(dx,dy,pw,ph) in enumerate(destinations,1):
                ax.text(dx+pw/2,dy+ph/2,f"C{number}",
                        ha="center",va="center",fontsize=8,fontweight="bold",
                        color="black",clip_on=True,
                        bbox=dict(facecolor="white",edgecolor="black",
                                  alpha=0.88,boxstyle="round,pad=0.18"))
            fig.tight_layout()
            plt.show()
            # The opposite edge from the anchor is where the cut tile appears.
            print(f"FLOOR: {Wmm:g} × {Hmm:g} mm | TILE: {Tw:g} × {Th:g} mm | GROUT: {G:g} mm")
            print(f"LAYOUT: {corner.value} | {len(xs)} columns × {len(ys)} rows")
            print(f"Each full tile is {Tw:g} × {Th:g} mm; each interior grout joint is {G:g} mm.")
            for axis_name, pieces, axis_tile_size in [("X", xs, Tw), ("Y", ys, Th)]:
                print(f"{axis_name} tile ranges (mm):", ", ".join(f"{a:g}–{b:g}" for a,b,*_ in pieces))
                for side, piece in [("first",pieces[0]),("last",pieces[-1])]:
                    width=piece[1]-piece[0]
                    if width<axis_tile_size-1e-7:
                        print(f"  {axis_name} {side} boundary cut: {width:g} mm (trim {axis_tile_size-width:g} mm)")
            print(f"Top-left piece: {xs[0][1]-xs[0][0]:g} × {ys[0][1]-ys[0][0]:g} mm")
            print(f"Bottom-right piece: {xs[-1][1]-xs[-1][0]:g} × {ys[-1][1]-ys[-1][0]:g} mm")
            print("\nPATTERN-AWARE TILE QUANTITY (including 0°/90°/180°/270° rotations)")
            print(f"Full uncut tiles: {full}")
            print(f"Cut pieces required: {cut_pieces}")
            print(f"Additional whole tiles for cuts: {cut_tiles}")
            print(f"Cut pieces taken from existing offcuts: {reused}")
            print(f"Pieces installed after rotation: {rotated}")
            print(f"TOTAL WHOLE TILES NEEDED: {total}")
            print(f"Estimated leftover tile material: {waste/1_000_000:.3f} m²")
            print("Estimate assumes no blade loss or breakage; 0°, 90°, 180° and 270° rotations allowed.")
            print("Offcuts must fit physically AND match the visible pattern.")
            print("Pattern tolerance: 24/255 mean RGB difference (JPEG imperfections allowed).")
            print("CUTTING PLAN (tile number among additional cut tiles; source x/y in mm):")
            for i,((tid,px,py,pw,ph,angle),(dx,dy,dw,dh)) in enumerate(zip(plan,destinations),1):
                print(f"  C{i}: source tile {tid} at ({px:g},{py:g}) mm, "
                      f"cut {pw:g} × {ph:g} mm, rotate {angle}°, "
                      f"INSTALL at floor ({dx:g},{dy:g}) mm")
            print("This is a cutting estimate, not an optimized or guaranteed minimum.")

        except Exception as e:
            print("Error:",e)

button.on_click(render)
display(widgets.VBox([
    widgets.HTML("<h3>Tile Visualizer</h3><b>Floor dimensions</b>"),
    fwui,fhui,widgets.HTML("<b>Tile dimensions</b>"),
    twui,thui,widgets.HTML("<b>Grout</b>"),
    gwui,color,corner,upload,button,out
]))
